# Algorithmes d'optimisation sans contrainte : implémentation *from scratch*

Ce notebook implémente **à la main** **sans aucune bibliothèque de calcul** : uniquement des listes Python et le module standard `math` (pour `sqrt`). Les produits scalaires, normes, produits matriciels et la résolution de systèmes linéaires sont codés à la main. 

1. l'**algorithme de Newton** ;
2. l'**algorithme quasi-Newton** (méthode BFGS) ;
3. l'**algorithme du gradient à pas fixe** ;
4. l'**algorithme du gradient à pas optimal**.

Tous ces algorithmes suivent le même schéma itératif :

$$x_{k+1} = x_k + \alpha_k\, d_k$$

où $d_k$ est la **direction de descente** et $\alpha_k>0$ le **pas**. Ils diffèrent par le choix de $(d_k,\alpha_k)$.

| Algorithme | Direction $d_k$ | Pas $\alpha_k$ |
|---|---|---|
| Gradient à pas fixe | $-\nabla f(x_k)$ | constant |
| Gradient à pas optimal | $-\nabla f(x_k)$ | $\arg\min_{\alpha>0} f(x_k+\alpha d_k)$ |
| Newton | $-\nabla^2 f(x_k)^{-1}\nabla f(x_k)$ | 1 |
| Quasi-Newton (BFGS) | $-H_k\nabla f(x_k)$ avec $H_k\approx\nabla^2 f(x_k)^{-1}$ | recherche linéaire (Armijo) |

**Critère d'arrêt** commun : $\|\nabla f(x_k)\|<\varepsilon$ (le gradient est quasi nul : on est proche d'un point critique).

## 0. Outils d'algèbre linéaire codés à la main et problème test

On choisit une **fonction quadratique** de $\mathbb{R}^2$ pour laquelle on peut tout calculer à la main :

$$f(x)=\tfrac12 x^\top A x - b^\top x,\qquad A=\begin{pmatrix}10&2\\2&1\end{pmatrix},\quad b=\begin{pmatrix}1\\2\end{pmatrix}.$$

- Gradient : $\nabla f(x)=Ax-b$ ; Hessienne : $\nabla^2 f(x)=A$ (constante).
- $A$ est symétrique définie positive (valeurs propres $\lambda_{\min}\approx 0.576$ et $\lambda_{\max}\approx 10.424$) : $f$ est **strictement convexe** et possède un unique minimiseur, solution de $Ax^*=b$ :

$$x^*=A^{-1}b=\frac16\begin{pmatrix}1&-2\\-2&10\end{pmatrix}\begin{pmatrix}1\\2\end{pmatrix}=\begin{pmatrix}-0.5\\3\end{pmatrix},\qquad f(x^*)=-2.75 .$$



Point de départ commun : $x_0=(2,2)^\top$. En ce point : $\nabla f(x_0)=Ax_0-b=\begin{pmatrix}24\\6\end{pmatrix}-\begin{pmatrix}1\\2\end{pmatrix}=\begin{pmatrix}23\\4\end{pmatrix}$.

In [2]:
import math                                   # module standard : on s'en sert uniquement pour la racine carrée
import matplotlib.pyplot as plt                 # utilisé SEULEMENT pour dessiner les graphiques (pas dans les algorithmes)

# =====================  OUTILS D'ALGÈBRE LINÉAIRE FAITS MAISON  =====================
# Convention : un vecteur est une liste de nombres, une matrice est une liste de lignes.

def dot(u, v):                                  # produit scalaire u^T v
    s = 0.0                                     # accumulateur de la somme
    for i in range(len(u)):                     # parcours de toutes les composantes
        s += u[i] * v[i]                        # on ajoute le produit u_i * v_i
    return s                                    # retourne le produit scalaire

def norm(u):                                    # norme euclidienne ||u||
    return math.sqrt(dot(u, u))                 # racine carrée de u^T u

def add(u, v):                                  # somme de deux vecteurs
    return [u[i] + v[i] for i in range(len(u))] # addition composante par composante

def sub(u, v):                              # différence u - v
    return [u[i] - v[i] for i in range(len(u))] # soustraction composante par composante

def scal(c, u):                                 # produit d'un scalaire c par un vecteur u
    return [c * ui for ui in u]                 # multiplication de chaque composante par c

def matvec(M, v):                               # produit matrice-vecteur M v
    return [dot(ligne, v) for ligne in M]       # chaque composante est le produit scalaire (ligne de M) . v

def matmul(M, N):                               # produit matriciel M N
    n, p, q = len(M), len(N), len(N[0])         # dimensions : M est n x p, N est p x q
    R = [[0.0] * q for _ in range(n)]           # matrice résultat n x q initialisée à zéro
    for i in range(n):                          # boucle sur les lignes de M
        for j in range(q):                      # boucle sur les colonnes de N
            for k in range(p):                  # somme sur l'indice commun
                R[i][j] += M[i][k] * N[k][j]    # R_ij = somme_k M_ik N_kj
    return R                                    # retourne le produit

def outer(u, v):                                # produit extérieur u v^T (matrice de terme général u_i v_j)
    return [[ui * vj for vj in v] for ui in u]  # une ligne par composante de u

def identite(n):                                # matrice identité n x n
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]  # 1 sur la diagonale, 0 ailleurs

def mat_add(M, N):                              # somme de deux matrices
    return [[M[i][j] + N[i][j] for j in range(len(M[0]))] for i in range(len(M))]  # terme à terme

def mat_sub(M, N):                              # différence de deux matrices
    return [[M[i][j] - N[i][j] for j in range(len(M[0]))] for i in range(len(M))]  # terme à terme

def mat_scal(c, M):                             # produit d'un scalaire par une matrice
    return [[c * mij for mij in ligne] for ligne in M]  # chaque coefficient est multiplié par c

def resoudre(M, b):                             # résout M x = b par élimination de Gauss avec pivot partiel
    n = len(b)                                  # taille du système
    T = [M[i][:] + [b[i]] for i in range(n)]    # matrice augmentée [M | b] (copie pour ne pas modifier M)
    for k in range(n):                          # pour chaque colonne k (étape d'élimination)
        p = max(range(k, n), key=lambda i: abs(T[i][k]))  # ligne du plus grand pivot en valeur absolue
        T[k], T[p] = T[p], T[k]                 # on échange la ligne k et la ligne du pivot
        if abs(T[k][k]) < 1e-14:                # pivot quasi nul : la matrice est singulière
            raise ValueError("Matrice singulière : système impossible à résoudre")  # on signale l'erreur
        for i in range(k + 1, n):               # on annule les coefficients sous le pivot
            m = T[i][k] / T[k][k]               # multiplicateur de la ligne i
            for j in range(k, n + 1):           # mise à jour de toute la ligne i
                T[i][j] -= m * T[k][j]          # L_i <- L_i - m * L_k
    x = [0.0] * n                               # vecteur solution
    for i in range(n - 1, -1, -1):              # remontée triangulaire, de la dernière ligne à la première
        s = sum(T[i][j] * x[j] for j in range(i + 1, n))  # contribution des inconnues déjà calculées
        x[i] = (T[i][n] - s) / T[i][i]          # x_i = (second membre - s) / pivot
    return x                                    # retourne la solution

# =====================  PROBLÈME TEST (FONCTION QUADRATIQUE)  =====================
A = [[10.0, 2.0],                               # première ligne de la matrice A
     [2.0, 1.0]]                                # seconde ligne de A (A symétrique définie positive)
b = [1.0, 2.0]                                  # vecteur b du terme linéaire

def f(x):                                       # fonction objectif f(x) = 1/2 x^T A x - b^T x
    return 0.5 * dot(x, matvec(A, x)) - dot(b, x)  # produit scalaire de x avec A x, moins b^T x

def grad_f(x):                                  # gradient de f
    return sub(matvec(A, x), b)                 # grad f(x) = A x - b

def hess_f(x):                                  # hessienne de f
    return [ligne[:] for ligne in A]            # constante égale à A (on retourne une copie)

# ---------- Paramètres communs ----------
x0 = [2.0, 2.0]                                 # point de départ de tous les algorithmes
tol = 1e-6                                      # tolérance sur la norme du gradient (critère d'arrêt)
max_iter = 10000                                # nombre maximal d'itérations (garde-fou)

x_star = resoudre(A, b)                         # solution exacte de A x = b (sert uniquement de référence)
tr = A[0][0] + A[1][1]                          # trace de A
det = A[0][0] * A[1][1] - A[0][1] * A[1][0]     # déterminant de A
lam_min = (tr - math.sqrt(tr**2 - 4 * det)) / 2 # plus petite valeur propre (formule du polynôme caractéristique 2x2)
lam_max = (tr + math.sqrt(tr**2 - 4 * det)) / 2 # plus grande valeur propre
print("Minimiseur exact x* =", x_star)          # attendu : (-0.5, 3)
print("f(x*) =", f(x_star))                     # attendu : -2.75
print("Valeurs propres de A =", lam_min, lam_max)  # attendu : 0.5756 et 10.4244

Minimiseur exact x* = [-0.5, 3.0]
f(x*) = -2.75
Valeurs propres de A = 0.575571099101948 10.424428900898052


## 1. Algorithme de Newton

### Principe
On remplace $f$ par son **développement de Taylor d'ordre 2** en $x_k$ :

$$f(x_k+d)\approx f(x_k)+\nabla f(x_k)^\top d+\tfrac12 d^\top \nabla^2 f(x_k)\, d .$$

Si $\nabla^2 f(x_k)$ est définie positive, ce modèle quadratique est minimisé en $d_k$ solution de

$$\nabla^2 f(x_k)\, d_k=-\nabla f(x_k)\quad\Longrightarrow\quad x_{k+1}=x_k-\nabla^2 f(x_k)^{-1}\nabla f(x_k).$$

### Exemple numérique (une itération sur notre problème)
Avec $x_0=(2,2)^\top$, $\nabla f(x_0)=(23,4)^\top$ et $\nabla^2 f=A$ :

$$A^{-1}\nabla f(x_0)=\frac16\begin{pmatrix}1&-2\\-2&10\end{pmatrix}\begin{pmatrix}23\\4\end{pmatrix}=\frac16\begin{pmatrix}15\\-6\end{pmatrix}=\begin{pmatrix}2.5\\-1\end{pmatrix}$$

$$x_1=\begin{pmatrix}2\\2\end{pmatrix}-\begin{pmatrix}2.5\\-1\end{pmatrix}=\begin{pmatrix}-0.5\\3\end{pmatrix}=x^* .$$



In [3]:
def newton(grad, hess, x0, tol=1e-6, max_iter=100):
    """Algorithme de Newton pur (pas = 1).
    grad : fonction qui retourne le gradient en x
    hess : fonction qui retourne la hessienne en x
    x0 : point initial ; tol : tolérance sur ||grad|| ; max_iter : nombre max d'itérations
    Retourne : (dernier itéré, historique des itérés, historique des normes du gradient)
    """
    x = list(x0)                                # copie de x0 pour ne pas modifier l'original
    xs = [x[:]]                                 # liste qui mémorise tous les itérés (pour tracer ensuite)
    gnorms = [norm(grad(x))]                    # liste qui mémorise la norme du gradient à chaque itération
    for k in range(max_iter):                   # boucle principale sur les itérations
        g = grad(x)                             # gradient au point courant
        if norm(g) < tol:                       # critère d'arrêt : gradient presque nul
            break                               # on sort de la boucle : convergence atteinte
        H = hess(x)                             # hessienne au point courant
        d = resoudre(H, scal(-1.0, g))          # direction de Newton : résout H d = -g (élimination de Gauss maison)
        x = add(x, d)                           # mise à jour de l'itéré avec un pas égal à 1
        xs.append(x[:])                         # on mémorise le nouvel itéré
        gnorms.append(norm(grad(x)))            # on mémorise la norme du gradient au nouvel itéré
    return x, xs, gnorms                        # retourne les résultats

# ---------- Test sur la fonction quadratique ----------
x_newton, xs_newton, g_newton = newton(grad_f, hess_f, x0, tol)    # exécution de Newton
print("Newton : x =", x_newton)                                    # solution trouvée
print("Nombre d'itérations =", len(xs_newton) - 1)                 # attendu : 1
print("Erreur ||x - x*|| =", norm(sub(x_newton, x_star)))          # distance à la solution exacte

Newton : x = [-0.5, 3.000000000000001]
Nombre d'itérations = 1
Erreur ||x - x*|| = 8.881784197001252e-16


## 2. Algorithme du gradient à pas fixe

### Principe
On avance dans la direction de plus forte descente $d_k=-\nabla f(x_k)$ avec un pas **constant** $\alpha$ :

$$x_{k+1}=x_k-\alpha\,\nabla f(x_k).$$

### Choix du pas
Pour une fonction quadratique, l'erreur $e_k=x_k-x^*$ vérifie $e_{k+1}=(I-\alpha A)e_k$. La méthode converge si et seulement si toutes les valeurs propres de $I-\alpha A$ sont de module $<1$, soit

$$0<\alpha<\frac{2}{\lambda_{\max}(A)}\approx\frac{2}{10.424}\approx 0.192 .$$

Le taux de convergence est $\rho=\max_i|1-\alpha\lambda_i|$. Avec $\alpha=0.1$ :
- $|1-0.1\times 10.424|\approx 0.042$ (direction « raide », convergence très rapide) ;
- $|1-0.1\times 0.576|\approx 0.942$ (direction « plate », convergence lente) ;

donc $\rho\approx0.942$ : on gagne seulement ~6 % de précision par itération, d'où un nombre d'itérations de l'ordre de $\ln(10^{-6})/\ln(0.942)\approx 230$.

### Exemple numérique (première itération, $\alpha=0.1$)
$$x_1=\begin{pmatrix}2\\2\end{pmatrix}-0.1\begin{pmatrix}23\\4\end{pmatrix}=\begin{pmatrix}-0.3\\1.6\end{pmatrix}.$$

Si on prenait $\alpha>0.192$ (par exemple $0.25$), la composante selon la direction raide serait multipliée par $|1-0.25\times10.424|\approx1.6>1$ : l'algorithme **diverge**.

In [4]:
def gradient_pas_fixe(grad, x0, alpha, tol=1e-6, max_iter=10000):
    """Descente de gradient à pas constant alpha.
    grad : fonction qui retourne le gradient ; x0 : point initial ; alpha : pas constant
    Retourne : (dernier itéré, historique des itérés, historique des normes du gradient)
    """
    x = list(x0)                                # copie de x0
    xs = [x[:]]                                 # historique des itérés
    gnorms = [norm(grad(x))]                    # historique des normes du gradient
    for k in range(max_iter):                   # boucle principale
        g = grad(x)                             # gradient au point courant
        if norm(g) < tol:                       # critère d'arrêt : gradient presque nul
            break                               # convergence atteinte, on sort
        x = sub(x, scal(alpha, g))              # mise à jour : x <- x - alpha * g
        xs.append(x[:])                         # on mémorise le nouvel itéré
        gnorms.append(norm(grad(x)))            # on mémorise la norme du gradient
        if gnorms[-1] > 1e100:                  # détection de divergence (norme gigantesque)
            print("Divergence détectée à l'itération", k + 1)  # message d'avertissement
            break                               # on arrête l'algorithme
    return x, xs, gnorms                        # retourne les résultats

# ---------- Test avec alpha = 0.1 (pas valide car < 0.192) ----------
alpha_fixe = 0.1                                                      # pas constant choisi
x_fixe, xs_fixe, g_fixe = gradient_pas_fixe(grad_f, x0, alpha_fixe, tol, max_iter)  # exécution
print("Pas fixe : x =", x_fixe)                                       # solution trouvée
print("Nombre d'itérations =", len(xs_fixe) - 1)                      # de l'ordre de 230
print("Erreur ||x - x*|| =", norm(sub(x_fixe, x_star)))               # distance à la solution exacte
print("Premier itéré x1 =", xs_fixe[1])                               # doit valoir (-0.3, 1.6) comme calculé à la main

# ---------- Test avec un pas trop grand (alpha = 0.25) : divergence attendue ----------
_, _, g_div = gradient_pas_fixe(grad_f, x0, 0.25, tol, max_iter=60)   # on limite à 60 itérations
print("Norme du gradient après", len(g_div) - 1, "itérations avec alpha=0.25 :", g_div[-1])  # valeur énorme : divergence

Pas fixe : x = [-0.4999996489228936, 2.999998345649386]
Nombre d'itérations = 231
Erreur ||x - x*|| = 1.6911922090996617e-06
Premier itéré x1 = [-0.30000000000000027, 1.6]
Norme du gradient après 60 itérations avec alpha=0.25 : 51805315244272.48


## 3. Algorithme du gradient à pas optimal

### Principe
La direction est toujours $d_k=-\nabla f(x_k)$, mais à chaque itération le pas est choisi **optimalement** en minimisant la fonction d'une variable

$$\varphi_k(\alpha)=f\big(x_k-\alpha\nabla f(x_k)\big),\qquad \alpha_k=\arg\min_{\alpha>0}\varphi_k(\alpha).$$

On le calcule ici par une **recherche linéaire exacte** générale :
1. *encadrement* : on double un pas jusqu'à ce que $\varphi$ cesse de décroître, ce qui fournit un intervalle $[0,\alpha_{\max}]$ contenant un minimum ;
2. *section dorée* : on réduit l'intervalle d'un facteur $0.618$ à chaque évaluation, sans utiliser de dérivée.

### Cas quadratique : formule fermée (vérification)
Pour $f(x)=\tfrac12x^\top Ax-b^\top x$ et $g_k=\nabla f(x_k)$, $\varphi_k'(\alpha)=0$ donne

$$\alpha_k=\frac{g_k^\top g_k}{g_k^\top A\,g_k}.$$

### Exemple numérique (première itération)
Avec $g_0=(23,4)^\top$ : $g_0^\top g_0=529+16=545$, $Ag_0=(238,50)^\top$, $g_0^\top Ag_0=23\cdot238+4\cdot50=5674$, donc

$$\alpha_0=\frac{545}{5674}\approx0.0961,\qquad x_1=\begin{pmatrix}2\\2\end{pmatrix}-0.0961\begin{pmatrix}23\\4\end{pmatrix}\approx\begin{pmatrix}-0.209\\1.616\end{pmatrix}.$$

**Propriété caractéristique** : le pas optimal rend le nouveau gradient orthogonal à l'ancien, $\nabla f(x_{k+1})^\top\nabla f(x_k)=0$ ; les trajectoires font donc des **zigzags à angle droit**. La convergence est linéaire de taux $\left(\frac{\kappa-1}{\kappa+1}\right)^2$, soit environ $0.80$ dans le pire cas ici (le point de départ choisi est favorable : 9 itérations observées) : plus rapide que notre pas fixe, mais encore lente quand $\kappa$ est grand.

In [5]:
def section_doree(phi, a, b_, tol=1e-12):
    """Minimise une fonction phi unimodale d'une variable sur [a, b_] (méthode de la section dorée)."""
    invphi = (math.sqrt(5.0) - 1.0) / 2.0       # inverse du nombre d'or, environ 0.618
    c = b_ - invphi * (b_ - a)                  # premier point intérieur (côté gauche)
    d = a + invphi * (b_ - a)                   # second point intérieur (côté droit)
    fc, fd = phi(c), phi(d)                     # valeurs de phi aux deux points intérieurs
    while abs(b_ - a) > tol:                    # on réduit l'intervalle tant qu'il est plus grand que tol
        if fc < fd:                             # le minimum est dans [a, d]
            b_, d, fd = d, c, fc                # nouvelle borne droite = d ; l'ancien c devient le nouveau d
            c = b_ - invphi * (b_ - a)          # nouveau point intérieur gauche
            fc = phi(c)                         # une seule nouvelle évaluation de phi
        else:                                   # le minimum est dans [c, b_]
            a, c, fc = c, d, fd                 # nouvelle borne gauche = c ; l'ancien d devient le nouveau c
            d = a + invphi * (b_ - a)           # nouveau point intérieur droit
            fd = phi(d)                         # une seule nouvelle évaluation de phi
    return (a + b_) / 2.0                       # milieu de l'intervalle final = approximation du minimiseur

def pas_optimal(f, x, d):
    """Recherche linéaire exacte : retourne alpha minimisant phi(alpha) = f(x + alpha d)."""
    phi = lambda alpha: f(add(x, scal(alpha, d)))   # fonction d'une variable à minimiser
    hi = 1e-4                                   # pas initial pour l'encadrement
    while phi(2 * hi) < phi(hi) and hi < 1e8:   # tant que phi décroît encore quand on double le pas
        hi *= 2.0                               # on double le pas
    return section_doree(phi, 0.0, 2 * hi)      # le minimum est dans [0, 2*hi] : on affine par section dorée

def gradient_pas_optimal(f, grad, x0, tol=1e-6, max_iter=10000):
    """Descente de gradient avec pas optimal (recherche linéaire exacte) à chaque itération.
    Retourne : (dernier itéré, historique des itérés, historique des normes du gradient, historique des pas)
    """
    x = list(x0)                                # copie de x0
    xs = [x[:]]                                 # historique des itérés
    gnorms = [norm(grad(x))]                    # historique des normes du gradient
    alphas = []                                 # historique des pas optimaux
    for k in range(max_iter):                   # boucle principale
        g = grad(x)                             # gradient au point courant
        if norm(g) < tol:                       # critère d'arrêt : gradient presque nul
            break                               # convergence atteinte
        d = scal(-1.0, g)                       # direction de plus forte descente
        alpha = pas_optimal(f, x, d)            # pas optimal le long de la direction d
        x = add(x, scal(alpha, d))              # mise à jour : x <- x + alpha d
        xs.append(x[:])                         # on mémorise le nouvel itéré
        gnorms.append(norm(grad(x)))            # on mémorise la norme du gradient
        alphas.append(alpha)                    # on mémorise le pas utilisé
    return x, xs, gnorms, alphas                # retourne les résultats

# ---------- Test sur la fonction quadratique ----------
x_opt, xs_opt, g_opt, alphas_opt = gradient_pas_optimal(f, grad_f, x0, tol, max_iter)  # exécution
print("Pas optimal : x =", x_opt)                                    # solution trouvée
print("Nombre d'itérations =", len(xs_opt) - 1)                      # nombre d'itérations
print("Erreur ||x - x*|| =", norm(sub(x_opt, x_star)))               # distance à la solution exacte
g0 = grad_f(x0)                                                      # gradient initial (23, 4)
alpha_formule = dot(g0, g0) / dot(g0, matvec(A, g0))                 # formule fermée 545/5674 (cas quadratique)
print("alpha_0 (section dorée) =", alphas_opt[0])                    # premier pas trouvé numériquement
print("alpha_0 (formule fermée) =", alpha_formule)                   # doit coïncider avec la ligne précédente
print("x1 =", xs_opt[1])                                             # doit valoir environ (-0.209, 1.616)
print("<grad(x1), grad(x0)> =", dot(grad_f(xs_opt[1]), g0))          # proche de 0 : gradients successifs orthogonaux

Pas optimal : x = [-0.4999999128330849, 2.999999544377253]
Nombre d'itérations = 9
Erreur ||x - x*|| = 4.638859329040411e-07
alpha_0 (section dorée) = 0.09605216810291695
alpha_0 (formule fermée) = 0.09605216778286922
x1 = [-0.20919986636708998, 1.6157913275883322]
<grad(x1), grad(x0)> = -1.815950804839872e-06


## 4. Algorithme quasi-Newton (BFGS)

### Principe
Newton est très rapide mais exige de calculer et résoudre avec la hessienne à chaque itération (coût $O(n^3)$ et dérivées secondes). Les méthodes **quasi-Newton** construisent à la place une approximation $H_k$ de **l'inverse** de la hessienne, uniquement à partir des gradients.

Direction : $d_k=-H_k\nabla f(x_k)$ ; pas $\alpha_k$ par recherche linéaire (condition d'Armijo, voir le code).

Après le pas on pose
$$s_k=x_{k+1}-x_k,\qquad y_k=\nabla f(x_{k+1})-\nabla f(x_k),\qquad \rho_k=\frac1{y_k^\top s_k}.$$

On impose l'**équation de la sécante** $H_{k+1}y_k=s_k$ (la nouvelle approximation reproduit la courbure observée), et la formule **BFGS** donne

$$H_{k+1}=\left(I-\rho_k s_k y_k^\top\right)H_k\left(I-\rho_k y_k s_k^\top\right)+\rho_k s_k s_k^\top .$$

Si $y_k^\top s_k>0$ (condition de courbure) et $H_k$ est définie positive, alors $H_{k+1}$ l'est aussi : $d_k$ reste une direction de descente. On saute la mise à jour si cette condition n'est pas vérifiée.

### Exemple numérique (première itération, $H_0=I$)
- $d_0=-g_0=(-23,-4)^\top$ : la première itération est un pas de gradient. Si le pas retenu est $\alpha_0\approx0.0961$, alors $s_0\approx(-2.209,-0.384)^\top$.
- $y_0=As_0\approx(-22.86,-4.80)^\top$ (cas quadratique : $y=As$), $y_0^\top s_0\approx52.3$, donc $\rho_0\approx0.0191$.
- $H_1$ n'est plus l'identité : elle contient de l'information sur la courbure, et vérifie $H_1y_0=s_0$.

À chaque itération, $H_k$ se rapproche de $A^{-1}$ : la convergence devient **superlinéaire** (plus rapide que le gradient, un peu moins que Newton) sans jamais calculer la hessienne.

In [6]:
def backtracking(f, x, d, g, alpha0=1.0, c=1e-4, rho=0.5):
    """Recherche linéaire à rebroussement (Armijo) : réduit alpha jusqu'à obtenir une décroissance suffisante.
    Condition d'Armijo : f(x + alpha d) <= f(x) + c * alpha * g^T d
    """
    alpha = alpha0                              # on commence avec le pas initial (1 pour les méthodes de type Newton)
    fx = f(x)                                   # valeur de f au point courant (calculée une seule fois)
    pente = dot(g, d)                           # dérivée directionnelle g^T d (négative si d est direction de descente)
    while f(add(x, scal(alpha, d))) > fx + c * alpha * pente and alpha > 1e-16:  # tant que la décroissance est insuffisante
        alpha *= rho                            # on divise le pas par 2
    return alpha                                # pas satisfaisant la condition d'Armijo

def quasi_newton_bfgs(f, grad, x0, tol=1e-6, max_iter=1000):
    """Algorithme quasi-Newton BFGS (approximation de l'inverse de la hessienne).
    Retourne : (dernier itéré, historique des itérés, historique des normes du gradient)
    """
    n = len(x0)                                 # dimension du problème
    I = identite(n)                             # matrice identité n x n
    x = list(x0)                                # copie de x0
    H = identite(n)                             # approximation initiale de l'inverse de la hessienne : H0 = I
    g = grad(x)                                 # gradient au point initial
    xs = [x[:]]                                 # historique des itérés
    gnorms = [norm(g)]                          # historique des normes du gradient
    for k in range(max_iter):                   # boucle principale
        if norm(g) < tol:                       # critère d'arrêt : gradient presque nul
            break                               # convergence atteinte
        d = scal(-1.0, matvec(H, g))            # direction quasi-Newton : d = -H g
        alpha = backtracking(f, x, d, g)        # pas par recherche linéaire d'Armijo
        x_new = add(x, scal(alpha, d))          # nouvel itéré
        g_new = grad(x_new)                     # gradient au nouvel itéré
        s = sub(x_new, x)                       # variation de x : s_k = x_{k+1} - x_k
        y = sub(g_new, g)                       # variation du gradient : y_k = g_{k+1} - g_k
        sy = dot(y, s)                          # produit scalaire y^T s (condition de courbure : doit être > 0)
        if sy > 1e-12:                          # on ne met à jour H que si la condition de courbure est vérifiée
            rho = 1.0 / sy                      # coefficient rho_k = 1 / (y^T s)
            gauche = mat_sub(I, mat_scal(rho, outer(s, y)))   # facteur de gauche : I - rho s y^T
            droite = mat_sub(I, mat_scal(rho, outer(y, s)))   # facteur de droite : I - rho y s^T
            H = mat_add(matmul(matmul(gauche, H), droite), mat_scal(rho, outer(s, s)))  # formule BFGS complète
        x, g = x_new, g_new                     # passage à l'itération suivante
        xs.append(x[:])                         # on mémorise le nouvel itéré
        gnorms.append(norm(g))                  # on mémorise la norme du gradient
    return x, xs, gnorms                        # retourne les résultats

# ---------- Test sur la fonction quadratique ----------
x_bfgs, xs_bfgs, g_bfgs = quasi_newton_bfgs(f, grad_f, x0, tol)    # exécution de BFGS
print("BFGS : x =", x_bfgs)                                        # solution trouvée
print("Nombre d'itérations =", len(xs_bfgs) - 1)                   # nombre d'itérations
print("Erreur ||x - x*|| =", norm(sub(x_bfgs, x_star)))            # distance à la solution exacte

BFGS : x = [-0.5000000031295527, 3.000000011500258]
Nombre d'itérations = 7
Erreur ||x - x*|| = 1.1918474573755206e-08
